In [0]:
# %pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
# Destino dos dados no volume do Unity Catalog.

import os
import sys

REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

os.environ["PIPELINE_DATA_ROOT"] = "/Volumes/avaliacao_sicredi/novarota/dados"

print(f"raiz do projeto: {REPO_ROOT}")

raiz do projeto: /Workspace/Users/monego.giovani@outlook.com/desafio-sicredi-cartoes


In [0]:
# Diagnostico do ambiente

from src.config import get_config_version, get_data_root, get_env, get_spark

spark = get_spark()

print(f"ambiente .......: {get_env()}")
print(f"raiz de dados ..: {get_data_root()}")
print(f"config version .: {get_config_version()}")
print(f"spark version ..: {spark.version}")
print(f"timezone .......: {spark.conf.get('spark.sql.session.timeZone')}")

ambiente .......: databricks
raiz de dados ..: /Volumes/avaliacao_sicredi/novarota/dados
config version .: 1.0.0
spark version ..: 4.2.0
timezone .......: UTC


In [0]:
# Geração da massa
from scripts.gerar_massa import gerar_massa

gerar_massa()

2026-11-05 19:03:13 | INFO    | scripts.gerar_massa                | inicio: geracao da massa (escala=1)
2026-11-05 19:03:13 | INFO    | scripts.gerar_massa                | destino=/Volumes/avaliacao_sicredi/novarota/dados/landing | semente=20261103 | 3 carga(s) entre 2026-08-14 e 2026-10-15
2026-11-05 19:03:13 | INFO    | scripts.gerar_massa                | landing/clientes.csv                           318 linhas
2026-11-05 19:03:13 | INFO    | scripts.gerar_massa                | landing/contas.csv                             340 linhas
2026-11-05 19:03:13 | INFO    | scripts.gerar_massa                | landing/cartoes.csv                            400 linhas
2026-11-05 19:03:14 | INFO    | scripts.gerar_massa                | landing/transacoes_cartao_20260814.csv        1809 linhas
2026-11-05 19:03:14 | INFO    | scripts.gerar_massa                | landing/transacoes_cartao_20260916.csv        2412 linhas
2026-11-05 19:03:14 | INFO    | scripts.gerar_massa                | la

PosixPath('/Volumes/avaliacao_sicredi/novarota/dados/landing')

In [0]:
# Pipeline completo
from src.pipeline import executar

resumo = executar()

2026-11-05 19:03:15 | INFO    | src.pipeline                       | inicio: pipeline completo
2026-11-05 19:03:15 | INFO    | src.ingestion.bronze               | inicio: camada bronze
2026-11-05 19:03:15 | INFO    | src.ingestion.bronze               | inicio: bronze.clientes
2026-11-05 19:03:19 | INFO    | src.ingestion.bronze               | clientes.csv -> bronze.clientes | data_referencia=2026-11-05 | 318 linhas
2026-11-05 19:03:19 | INFO    | src.ingestion.bronze               | fim: bronze.clientes em 3.7s
2026-11-05 19:03:19 | INFO    | src.ingestion.bronze               | inicio: bronze.contas
2026-11-05 19:03:21 | INFO    | src.ingestion.bronze               | contas.csv -> bronze.contas | data_referencia=2026-11-05 | 340 linhas
2026-11-05 19:03:21 | INFO    | src.ingestion.bronze               | fim: bronze.contas em 2.3s
2026-11-05 19:03:21 | INFO    | src.ingestion.bronze               | inicio: bronze.cartoes
2026-11-05 19:03:23 | INFO    | src.ingestion.bronze          

In [0]:
# Conferência das tabelas materializadas
from src.config import resolve_target

for tabela in (
    "gold_transacoes_cartao",
    "gold_resumo_cliente_mes",
    "gold_ranking_estabelecimento",
    "gold_alertas_qualidade",
):
    linhas = spark.read.format("delta").load(resolve_target("gold", tabela)).count()
    print(f"{tabela:32s} {linhas:7d} linhas")

gold_transacoes_cartao              5659 linhas
gold_resumo_cliente_mes              463 linhas
gold_ranking_estabelecimento          80 linhas
gold_alertas_qualidade                11 linhas


In [0]:
# Resumo de qualidade
display(
    spark.read.format("delta")
    .load(resolve_target("gold", "gold_alertas_qualidade"))
    .orderBy("entidade", "tipo")
)

entidade,tipo,motivo,qtd_registros
cartoes,alerta,bandeira_fora_do_dominio,7
cartoes,rejeicao,fk_invalida_id_conta,12
clientes,alerta,estado_uf_invalida,6
clientes,rejeicao,cpf_formato_invalido,7
clientes,rejeicao,cpf_digito_verificador_invalido,18
contas,rejeicao,fk_invalida_id_cliente,9
contas,rejeicao,status_conta_fora_do_dominio,8
estornos,rejeicao,fk_invalida_id_transacao,14
transacoes,rejeicao,valor_nao_positivo,19
transacoes,rejeicao,fk_invalida_id_cartao,12


In [0]:
# Consultas analíticas
from pathlib import Path

from src.transformations.gold import registrar_views, registrar_views_gold

registrar_views()
registrar_views_gold()

for arquivo in sorted(Path("sql/analises").glob("*.sql")):
    print(f"\n{'=' * 72}\n{arquivo.name}\n{'=' * 72}")
    display(spark.sql(arquivo.read_text(encoding="utf-8")))


crescimento_mensal_clientes.sql


id_cliente,nome_cliente,mes_anterior,mes_atual,valor_mes_anterior,valor_mes_atual,variacao_absoluta,crescimento_percentual,qtd_transacoes
CLI000132,Olivia Silva,2026-08,2026-09,616.07,7174.07,6558.00,1064.49,5
CLI000098,Renata Ferreira,2026-09,2026-10,747.82,7495.90,6748.08,902.37,3
CLI000023,Felipe Costa,2026-08,2026-09,783.12,7248.32,6465.20,825.57,6
CLI000111,Lucas Santos,2026-08,2026-09,1556.87,13818.41,12261.54,787.58,8
CLI000289,Renata Souza,2026-08,2026-09,3027.95,22282.78,19254.83,635.90,12
CLI000153,Isabel Santos,2026-08,2026-09,3959.49,23135.65,19176.16,484.31,14
CLI000227,Gabriela Lima,2026-08,2026-09,5271.70,26446.05,21174.35,401.66,13
CLI000168,Vitor Silva,2026-08,2026-09,1323.71,5541.75,4218.04,318.65,4
CLI000264,Henrique Ferreira,2026-08,2026-09,10457.30,43198.84,32741.54,313.10,18
CLI000005,Felipe Costa,2026-08,2026-09,4324.50,15599.56,11275.06,260.73,11



evolucao_mensal_cliente.sql


id_cliente,nome_cliente,ano_mes,numero_do_mes,valor_mes_anterior,valor_mes_atual,valor_mes_seguinte,qtd_transacoes,ticket_medio,tendencia,variacao_percentual
CLI000005,Felipe Costa,2026-08,1,null,4324.50,15599.56,3,1441.50,primeiro mes,null
CLI000005,Felipe Costa,2026-09,2,4324.50,15599.56,6738.35,11,1733.28,crescimento,260.73
CLI000005,Felipe Costa,2026-10,3,15599.56,6738.35,null,2,3369.18,retracao,-56.80
CLI000009,Paulo Silva,2026-08,1,null,13107.17,30015.02,10,1310.72,primeiro mes,null
CLI000009,Paulo Silva,2026-09,2,13107.17,30015.02,17031.97,18,1667.50,crescimento,129.00
CLI000009,Paulo Silva,2026-10,3,30015.02,17031.97,null,7,2433.14,retracao,-43.26
CLI000010,Vitor Ribeiro,2026-08,1,null,15929.69,20014.16,9,1769.97,primeiro mes,null
CLI000010,Vitor Ribeiro,2026-09,2,15929.69,20014.16,10654.91,13,1539.55,crescimento,25.64
CLI000010,Vitor Ribeiro,2026-10,3,20014.16,10654.91,null,9,1331.86,retracao,-46.76
CLI000012,Sergio Pereira,2026-08,1,null,12111.83,14617.10,6,2018.64,primeiro mes,null



quartis_clientes.sql


id_cliente,nome_cliente,quartil,faixa,posicao,valor_total,qtd_transacoes,ticket_medio,valor_estornado,meses_ativos
CLI000138,Renata Souza,1,Q1 - maior movimentacao,1,188694.21,108,1747.17,4039.44,3
CLI000076,Joao Silva,1,Q1 - maior movimentacao,2,160162.14,91,1760.02,26.79,3
CLI000164,Karina Silva,1,Q1 - maior movimentacao,3,159877.59,99,1614.93,10041.03,3
CLI000205,Elisa Ferreira,1,Q1 - maior movimentacao,4,152469.93,88,1732.61,12717.18,3
CLI000163,Tatiana Pereira,1,Q1 - maior movimentacao,5,151461.93,96,1577.73,4625.49,3
CLI000206,Sergio Ferreira,1,Q1 - maior movimentacao,6,150815.88,95,1587.54,12535.46,3
CLI000276,Diego Alves,1,Q1 - maior movimentacao,7,149042.26,88,1693.66,4054.35,3
CLI000102,Lucas Costa,1,Q1 - maior movimentacao,8,145327.04,79,1839.58,0.00,3
CLI000024,Lucas Souza,1,Q1 - maior movimentacao,9,137200.25,77,1781.82,802.97,3
CLI000244,Nelson Rodrigues,1,Q1 - maior movimentacao,10,136293.70,67,2034.23,34.89,3



ranking_estabelecimentos_por_cidade.sql


cidade_estabelecimento,posicao,posicao_com_empate,nome_estabelecimento,mcc,valor_liquido,participacao_na_cidade,qtd_transacoes,qtd_clientes,qtd_estornadas
Belo Horizonte,1,1,Eletronicos Tech,5732,146735.88,11.93,78,57,1
Belo Horizonte,2,2,Loja de Conveniencia,5999,141119.94,11.48,86,56,5
Belo Horizonte,3,3,Casa de Apostas,7995,140810.37,11.45,77,53,4
Campinas,1,1,Casa de Apostas,7995,133708.52,11.70,80,57,6
Campinas,2,2,Mercado Bom Preco,5411,121761.16,10.66,61,43,2
Campinas,3,3,Cantina Italiana,5812,118810.47,10.40,68,50,2
Canoas,1,1,Eletronicos Tech,5732,184361.31,15.04,87,65,2
Canoas,2,2,Transporte Urbano,4121,137109.85,11.19,77,55,0
Canoas,3,3,Mercado Bom Preco,5411,130977.36,10.69,80,55,1
Caxias do Sul,1,1,Restaurante da Esquina,5812,137712.03,11.43,73,56,2


In [0]:
# Registro no catalogo
from src.config import resolve_target

CATALOGO = "avaliacao_sicredi"
SCHEMA = f"{CATALOGO}.novarota_gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")

for tabela in (
    "gold_transacoes_cartao",
    "gold_resumo_cliente_mes",
    "gold_ranking_estabelecimento",
    "gold_alertas_qualidade",
):
    (
        spark.read.format("delta")
        .load(resolve_target("gold", tabela))
        .write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{SCHEMA}.{tabela}")
    )
    print(f"registrada: {SCHEMA}.{tabela}")

registrada: avaliacao_sicredi.novarota_gold.gold_transacoes_cartao
registrada: avaliacao_sicredi.novarota_gold.gold_resumo_cliente_mes
registrada: avaliacao_sicredi.novarota_gold.gold_ranking_estabelecimento
registrada: avaliacao_sicredi.novarota_gold.gold_alertas_qualidade


In [0]:
display(
    spark.sql(f"""
        SELECT nome_estabelecimento, cidade_estabelecimento,
               valor_liquido, rank_valor
        FROM {CATALOGO}.novarota_gold.gold_ranking_estabelecimento
        ORDER BY rank_valor
        LIMIT 10
    """)
)

nome_estabelecimento,cidade_estabelecimento,valor_liquido,rank_valor
Eletronicos Tech,Canoas,184361.31,1
Casa de Apostas,Florianopolis,152612.18,2
Eletronicos Tech,Belo Horizonte,146735.88,3
Supermercado Central,Sao Paulo,146544.98,4
Supermercado Central,Florianopolis,143993.86,5
Supermercado Central,Curitiba,141281.17,6
Transporte Urbano,Curitiba,141172.18,7
Loja de Conveniencia,Belo Horizonte,141119.94,8
Casa de Apostas,Belo Horizonte,140810.37,9
Restaurante da Esquina,Caxias do Sul,137712.03,10
